In [1]:
import os
%load_ext autoreload
%autoreload 2

In [2]:
os.chdir('..')

In [44]:
import asyncio
import logging
from datetime import datetime, timezone, timedelta

from src.services.news_dumper import TelegramDumper

In [47]:
import time

In [5]:
import asyncio
import logging
from datetime import datetime, timezone, timedelta, date
from collections import defaultdict
from typing import Dict, List

from src.services.news_dumper import TelegramDumper
from src.services.summarizer import NewsSummarizer
from src.models.news import NewsItem, Summary

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)

logger = logging.getLogger(__name__)

In [6]:
import asyncio
import logging
from datetime import datetime, timezone, timedelta, date
from collections import defaultdict
from typing import Dict, List

from src.services.news_dumper import TelegramDumper
from src.services.summarizer import NewsSummarizer
from src.models.news import NewsItem, Summary

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)

logger = logging.getLogger(__name__)


async def fetch_news_for_date_range(
    dumper: TelegramDumper,
    start_date: datetime,
    end_date: datetime
) -> List[NewsItem]:
    """Fetch all news items for a date range."""
    all_news_items = []
    
    # Iterate through each day from start_date to end_date (exclusive)
    current_date = start_date
    while current_date < end_date:
        logger.info(f"Fetching news for {current_date.date()}...")
        
        # Fetch news for the current date
        news_items = await dumper.dump_news_for_date(current_date)
        logger.info(f"Found {len(news_items)} news items for {current_date.date()}")
        
        # Add to collection
        all_news_items.extend(news_items)
        
        # Move to next day
        current_date += timedelta(days=1)
    
    return all_news_items


def group_news_by_date(news_items: List[NewsItem]) -> Dict[date, List[NewsItem]]:
    """Group news items by their date."""
    grouped = defaultdict(list)
    
    for item in news_items:
        # Extract date from datetime
        item_date = item.date.date() if isinstance(item.date, datetime) else item.date
        grouped[item_date].append(item)
    
    # Sort dates
    sorted_dates = sorted(grouped.keys())
    return {d: grouped[d] for d in sorted_dates}


async def generate_summaries_for_dates(
    summarizer: NewsSummarizer,
    grouped_news: Dict[date, List[NewsItem]]
) -> Dict[date, Summary]:
    """Generate summaries for each date."""
    summaries = {}
    
    for target_date, news_items in grouped_news.items():
        logger.info(f"\nGenerating summary for {target_date} ({len(news_items)} news items)...")
        
        if not news_items:
            logger.warning(f"No news items for {target_date}, skipping summary")
            continue
        
        summary = await summarizer.summarize_news(news_items, target_date)
        
        if summary:
            summaries[target_date] = summary
            logger.info(f"Successfully created summary for {target_date}")
        else:
            logger.error(f"Failed to create summary for {target_date}")
    
    return summaries


def print_summaries(summaries: Dict[date, Summary]):
    """Print all summaries in a formatted way."""
    print("\n" + "="*80)
    print("DAILY SUMMARIES")
    print("="*80 + "\n")
    
    for target_date in sorted(summaries.keys()):
        summary = summaries[target_date]
        print(f"{'='*80}")
        print(f"Date: {target_date}")
        print(f"News Count: {summary.news_count}")
        print(f"{'='*80}")
        
        if summary.key_topics:
            print("\nKey Topics:")
            for idx, topic in enumerate(summary.key_topics, 1):
                print(f"  {idx}. {topic}")
        else:
            print("\nNo key topics found.")
        
        print(f"\nSummary Text: {summary.summary_text}")
        print("\n")


async def main():
    """Main function to fetch news and generate summaries."""
    # Define date range
    start_date = datetime(2026, 1, 1, tzinfo=timezone.utc)
    end_date = datetime(2026, 1, 10, tzinfo=timezone.utc)
    
    # Initialize services
    dumper = TelegramDumper()
    summarizer = NewsSummarizer()
    
    try:
        # Step 1: Fetch all news items
        logger.info(f"Fetching news from {start_date.date()} to {end_date.date()}...")
        all_news_items = await fetch_news_for_date_range(dumper, start_date, end_date)
        # all_news_items = news_items
        logger.info(f"\nTotal news items fetched: {len(all_news_items)}")
        
        # Step 2: Group news by date
        logger.info("\nGrouping news items by date...")
        grouped_news = group_news_by_date(all_news_items)
        logger.info(f"Found news for {len(grouped_news)} different dates")
        
        # Print grouping info
        print("\n" + "="*80)
        print("NEWS GROUPED BY DATE")
        print("="*80)
        for target_date in sorted(grouped_news.keys()):
            print(f"{target_date}: {len(grouped_news[target_date])} items")
        print("="*80 + "\n")
        
        # Step 3: Generate summaries for each date
        logger.info("\nGenerating summaries for each date...")
        summaries = await generate_summaries_for_dates(summarizer, grouped_news)
        
        # Step 4: Print summaries
        print_summaries(summaries)
        
        logger.info(f"\nCompleted! Generated {len(summaries)} summaries.")
        
        return summaries
        
    finally:
        # Always close the connection
        await dumper.close()


# summaries = await main()

In [72]:
start_date = datetime(2026, 1, 1, tzinfo=timezone.utc)
end_date = datetime(2026, 2, 28, tzinfo=timezone.utc)

# Initialize services
dumper = TelegramDumper()


# Step 1: Fetch all news items
logger.info(f"Fetching news from {start_date.date()} to {end_date.date()}...")
all_news_items = await fetch_news_for_date_range(dumper, start_date, end_date)
# all_news_items = news_items
logger.info(f"\nTotal news items fetched: {len(all_news_items)}")

# Step 2: Group news by date
logger.info("\nGrouping news items by date...")
grouped_news = group_news_by_date(all_news_items)
logger.info(f"Found news for {len(grouped_news)} different dates")

# Print grouping info
print("\n" + "="*80)
print("NEWS GROUPED BY DATE")
print("="*80)
for target_date in sorted(grouped_news.keys()):
    print(f"{target_date}: {len(grouped_news[target_date])} items")
print("="*80 + "\n")




2026-04-03 17:44:49,839 | INFO | research.summary_metrics | Fetching news from 2026-01-01 to 2026-02-28...
2026-04-03 17:44:49,840 | INFO | research.summary_metrics | Fetching news for 2026-01-01...
2026-04-03 17:44:49,841 | INFO | src.services.news_dumper | Dumping news for 2026-01-01
2026-04-03 17:44:49,842 | INFO | telethon.network.mtprotosender | Connecting to 149.154.167.92:443/TcpFull...
2026-04-03 17:44:49,896 | INFO | telethon.network.mtprotosender | Connection to 149.154.167.92:443/TcpFull complete!
2026-04-03 17:44:50,193 | INFO | src.services.news_dumper | Connected to Telegram
2026-04-03 17:44:50,554 | INFO | src.services.news_dumper | Found 1 messages for 2026-01-01
2026-04-03 17:44:50,555 | INFO | research.summary_metrics | Found 1 news items for 2026-01-01
2026-04-03 17:44:50,555 | INFO | research.summary_metrics | Fetching news for 2026-01-02...
2026-04-03 17:44:50,556 | INFO | src.services.news_dumper | Dumping news for 2026-01-02
2026-04-03 17:44:50,776 | INFO | src.s


NEWS GROUPED BY DATE
2026-01-01: 1 items
2026-01-02: 43 items
2026-01-03: 17 items
2026-01-04: 25 items
2026-01-05: 113 items
2026-01-06: 113 items
2026-01-07: 82 items
2026-01-08: 142 items
2026-01-09: 117 items
2026-01-10: 25 items
2026-01-11: 30 items
2026-01-12: 153 items
2026-01-13: 209 items
2026-01-14: 186 items
2026-01-15: 177 items
2026-01-16: 135 items
2026-01-17: 20 items
2026-01-18: 38 items
2026-01-19: 163 items
2026-01-20: 200 items
2026-01-21: 173 items
2026-01-22: 205 items
2026-01-23: 171 items
2026-01-24: 26 items
2026-01-25: 37 items
2026-01-26: 163 items
2026-01-27: 188 items
2026-01-28: 174 items
2026-01-29: 203 items
2026-01-30: 175 items
2026-01-31: 22 items
2026-02-01: 74 items
2026-02-02: 161 items
2026-02-03: 179 items
2026-02-04: 220 items
2026-02-05: 174 items
2026-02-06: 127 items
2026-02-07: 23 items
2026-02-08: 38 items
2026-02-09: 145 items
2026-02-10: 148 items
2026-02-11: 165 items
2026-02-12: 167 items
2026-02-13: 145 items
2026-02-14: 20 items
2026-

2026-04-03 18:59:50,542 | INFO | telethon.client.updates | Got difference for account updates


In [42]:
grouped_news2 = grouped_news

In [73]:
models = ['gpt-4.1-nano', 'gpt-4.1-mini', 'gpt-4o-mini']

model_to_summaries = {}

for model in models:
    
    summarizer = NewsSummarizer(model=model)
    
    # Step 3: Generate summaries for each date
    logger.info("\nGenerating summaries for each date...")
    summaries = await generate_summaries_for_dates(summarizer, grouped_news)

    model_to_summaries[model] = summaries
    


2026-04-03 17:46:26,340 | INFO | research.summary_metrics | 
Generating summaries for each date...
2026-04-03 17:46:26,342 | INFO | research.summary_metrics | 
Generating summary for 2026-01-01 (1 news items)...
2026-04-03 17:46:26,353 | INFO | src.services.summarizer | Calling OpenAI API to summarize 1 news items...
2026-04-03 17:46:29,713 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-03 17:46:29,726 | INFO | src.services.summarizer | Successfully created news summary
2026-04-03 17:46:29,729 | INFO | research.summary_metrics | Successfully created summary for 2026-01-01
2026-04-03 17:46:29,731 | INFO | research.summary_metrics | 
Generating summary for 2026-01-02 (43 news items)...
2026-04-03 17:46:29,761 | INFO | src.services.summarizer | Calling OpenAI API to summarize 43 news items...
2026-04-03 17:46:33,705 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-03 17:46:33,

In [74]:
import pickle

with open('summariesV2.pickle', 'wb') as f:
    pickle.dump(model_to_summaries, f)

In [75]:
model_to_summaries

{'gpt-4.1-nano': {datetime.date(2026, 1, 1): Summary(date=datetime.date(2026, 1, 1), news_count=1, key_topics=['Мировые финансовые рынки в основном закрыты в связи с празднованием Нового года, активность минимальна.', 'Центральные банки большинства стран не объявляли новых решений или изменений в монетарной политике на сегодняшний день.', 'Ожидается начало новых экономических отчетов и корпоративных публикаций в ближайшие дни, что может повлиять на динамику рынков.', 'Геополитическая ситуация остается стабильной, существенных событий, влияющих на рынки, не зафиксировано.', 'Мировая экономика продолжает восстанавливаться после предыдущих кризисных явлений, однако динамика остается умеренной.', 'Инвесторы ожидают новых данных по инфляции и росту ВВП, которые могут стать ключевыми в ближайшие недели.', 'Крупные корпорации готовятся к отчетным периодам, что может повлиять на их акции и рыночные индексы в будущем.', 'Объемы торгов на международных рынках минимальны из-за праздничных выходны

In [11]:
grouped_news

{datetime.date(2026, 1, 1): [NewsItem(message_id=353560, text='🎄**🗓КАЛЕНДАРЬ НА СЕГОДНЯ — 2026.01.01**\nВ большинстве стран мира торгов нет - Новый год', date=datetime.datetime(2026, 1, 1, 9, 11, 20, tzinfo=datetime.timezone.utc), views=78060, forwards=375)],
 datetime.date(2026, 1, 2): [NewsItem(message_id=353604, text='🚤#историяуспеха \nВ Гане арестован Ной, который строил ковчеги, чтобы спасти людей и зверей от Великого потопа 2.0, который должен был случиться 25 декабря 2025г. \n\nПророка арестовали после того, как потопа [не случилось](https://t.me/markettwits/353039), а на вырученные от поверивших в апокалипсис средства, Ной прикупил себе новый Мерседес.\n\nВ полиции Эбо Ноа говорит, что в последнюю минуту отговорил Бога от беды.', date=datetime.datetime(2026, 1, 2, 19, 18, 31, tzinfo=datetime.timezone.utc), views=88551, forwards=2237),
  NewsItem(message_id=353603, text='✴️#BTC #крипто #прогноз \nГлава исследовательского отдела Grayscale Зак Пандл в [интервью](https://www.cnbc.c

In [12]:
summaries

{datetime.date(2026, 1, 1): Summary(date=datetime.date(2026, 1, 1), news_count=1, key_topics=['Мировые финансовые рынки закрыты в связи с празднованием Нового года, активность минимальна.', 'Центральные банки большинства стран начали новый год без важных объявлений, ожидая экономических данных и решений в ближайшие недели.', 'Ожидается, что в январе 2026 года рынки продолжат адаптироваться к изменениям в глобальной экономической политике и монетарных стратегиях.', 'Геополитическая ситуация остается стабильной, без значимых конфликтов или событий, способных повлиять на мировые рынки в ближайшее время.', 'Мировая экономика демонстрирует признаки замедления, что может повлиять на корпоративные доходы в первом квартале 2026 года.', 'Инвесторы ожидают публикации важных экономических показателей за январь, включая данные по инфляции и ВВП в ключевых странах.', 'Крупные корпорации готовятся к отчетным сезонам, что может вызвать волатильность на фондовых рынках в ближайшие недели.', 'Обменные 

In [13]:
date = list(grouped_news.keys())[2]

In [14]:
date

datetime.date(2026, 1, 3)

In [15]:
grouped_news[date]

[NewsItem(message_id=353633, text='⚠️🇺🇸#сша #today \nПрезидент одной страны в возрасте почти 80 лет успешно прошел медобследование, сдал тест на оценку когнитивных способностей и сразу похитил президента другой страны, взяв на себя руководство его страной. Все это было сделано всего за несколько часов без каких либо потерь.\n\nМировое сообщество глубоко потрясено и выражает сильную озабоченность.\n\n[__mt в max__](https://max.ru/markettwits)', date=datetime.datetime(2026, 1, 3, 17, 13, 43, tzinfo=datetime.timezone.utc), views=89978, forwards=1780),
 NewsItem(message_id=353632, text='⚠️🇻🇪#венесуэла #геополитика \n**Новогодняя пресс-конференция Трампа: **\n\nМадуро – больше не президент.\n\nОперация США в Венесуэле прошла без потерь.\n\nСША возьмут на себя управление Венесуэлой до полной смены власти.\n\nСША останутся в Венесуэле столько, сколько потребуется.\n\nКрупные нефтяные компании США отправятся в Венесуэлу. \n\nСША готовы ко второй, гораздо более мощной волне операции в Венесуэле

In [19]:
news_texts = [item.text for item in grouped_news[date] if item.text.strip()]

# Limit text length
max_length = 8000
combined_text = "\n\n".join(news_texts)
if len(combined_text) > max_length:
    combined_text = combined_text[:max_length] + "..."
    news_texts = [combined_text]

In [20]:
news_texts

['⚠️🇺🇸#сша #today \nПрезидент одной страны в возрасте почти 80 лет успешно прошел медобследование, сдал тест на оценку когнитивных способностей и сразу похитил президента другой страны, взяв на себя руководство его страной. Все это было сделано всего за несколько часов без каких либо потерь.\n\nМировое сообщество глубоко потрясено и выражает сильную озабоченность.\n\n[__mt в max__](https://max.ru/markettwits)',
 '⚠️🇻🇪#венесуэла #геополитика \n**Новогодняя пресс-конференция Трампа: **\n\nМадуро – больше не президент.\n\nОперация США в Венесуэле прошла без потерь.\n\nСША возьмут на себя управление Венесуэлой до полной смены власти.\n\nСША останутся в Венесуэле столько, сколько потребуется.\n\nКрупные нефтяные компании США отправятся в Венесуэлу. \n\nСША готовы ко второй, гораздо более мощной волне операции в Венесуэле, если понадобится. \n\nМадуро и его жена вскоре предстанут перед судом во всей полноте американского правосудия и пройдут суд на американской земле.\n\nЭмбарго на нефть ост

In [23]:
import os
import json
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def call_judge_llm(prompt):
    try:
        response = client.chat.completions.create(
            model="gpt-4o", # Для оценки лучше использовать gpt-4o, она строже
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        return {"score": 0, "reasoning": f"Error: {str(e)}"}

In [24]:
def calculate_relevance(original_news_180, summary_10):
    prompt = f"""
    Дан полный список новостей: {original_news_180}
    Дан итоговый топ-10: {summary_10}
    
    Оцени по шкале от 1 до 10, насколько выбранные 10 новостей являются 
    самыми важными событиями дня. Игнорируй стиль письма, оценивай только выбор тем.
    Выдай ответ в формате JSON: {{"score": float, "reasoning": str}}
    """
    response = call_judge_llm(prompt) # Вызов GPT-4o или Claude 3.5
    return response

In [26]:
text_summary = '\n'.join(summaries[date].key_topics)
print(
    text_summary
)

Масштабные военные действия США в Венесуэле: удары по Каракасу, захват Мадуро и его жены, начало наземных операций, что вызывает глобальную геополитическую нестабильность и влияет на нефтяные рынки.
Глубокий кризис в Венесуэле: США заявляют о полном контроле и намерениях управлять страной до смены власти, что вызывает озабоченность международного сообщества и может привести к росту цен на нефть.
Международная реакция на военные действия: Куба, Турция, Колумбия и другие страны требуют срочной реакции, а Россия обещает заявление, что усиливает геополитическую напряженность.
Обострение конфликта между США и Китаем: возможное ухудшение отношений из-за энергетической зависимости Китая от Венесуэлы и Ирана, что может повлиять на глобальные энергетические рынки.
Объявление о новых санкциях и эмбарго на нефть Венесуэлы: сохранение ограничений, что поддерживает цены на нефть на высоких уровнях и создает неопределенность для нефтяных компаний.
Обострение ситуации в Тайваньском проливе: продажа о

In [27]:
calculate_relevance(news_texts, text_summary)

2026-04-02 21:53:53,708 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{'score': 9.0,
 'reasoning': 'Выбранные 10 новостей охватывают ключевые аспекты текущей геополитической ситуации, включая масштабные военные действия США в Венесуэле, международную реакцию, влияние на энергетические рынки и глобальную стабильность. Эти темы действительно являются важными событиями дня, так как они затрагивают международные отношения, экономику и безопасность. Единственное, что можно было бы улучшить, это добавить больше контекста о реакции внутри США и других крупных мировых держав, таких как ЕС, чтобы получить более полную картину глобальной реакции.'}

In [59]:
def evaluate_faithfulness(original_news, summary_10):
    prompt = f"""
    Ты — фактчекер. Сравни итоговый ТОП-10 с исходными новостями. 
    Твоя цель: найти любые искажения, ошибки в именах, цифрах или выдуманные факты (галлюцинации).
    
    Оценка 10: Всё строго соответствует исходнику.
    Оценка 0: В ТОП-10 есть факты, которых не было в источнике.
    
    ИСХОДНЫЕ ДАННЫЕ: {original_news}
    ТОП-10 ДЛЯ ПРОВЕРКИ: {summary_10}
    
    Верни ответ строго в формате JSON: {{"score": float, "reasoning": "список найденных неточностей или подтверждение их отсутствия"}}
    """
    return call_judge_llm(prompt)

In [60]:
evaluate_faithfulness(news_texts, text_summary)

2026-03-10 22:31:15,693 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{'score': 8.0,
 'reasoning': "ТОП-10 в основном соответствует исходным данным, но есть некоторые неточности и обобщения. 1) В исходных данных не упоминается 'глобальная геополитическая нестабильность', хотя это подразумевается. 2) В ТОП-10 говорится о 'возможных санкциях и ограничениях на нефтяной сектор', что не было явно указано в исходных данных. 3) Упоминание о 'возможных перебоях в поставках нефти' и 'росте цен' также не было явно указано, хотя это логическое следствие. 4) В ТОП-10 говорится о 'внутренних заявлениях и угрозах со стороны США и их союзников', что не было явно указано в исходных данных. В остальном, основные события и реакции, описанные в ТОП-10, соответствуют исходным данным."}

In [61]:
def evaluate_redundancy(summary_10):
    prompt = f"""
    Проанализируй список из 10 новостей на предмет смысловых дублей.
    Если две или более новости описывают одно и то же событие — снижай балл.
    
    Оценка 10: Все новости уникальны.
    Оценка 5: Есть 1-2 дублирующих темы.
    Оценка 0: Список состоит из повторов.
    
    ТОП-10 ДЛЯ АНАЛИЗА: {summary_10}
    
    Верни ответ строго в формате JSON: {{"score": float, "reasoning": "какие новости дублируют друг друга"}}
    """
    return call_judge_llm(prompt)

In [63]:
evaluate_redundancy(text_summary)

2026-03-10 22:32:01,123 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{'score': 0.0,
 'reasoning': 'Все новости связаны с военной эскалацией в Венесуэле и ее влиянием на глобальные рынки и геополитическую ситуацию. Они описывают одно и то же событие с разных точек зрения: военные действия, реакции рынков, международные реакции и политические последствия. Таким образом, все новости являются смысловыми дублями.'}

In [57]:
summarizer.model

'gpt-4.1-nano'

In [12]:
from research.summary_metrics import *

In [34]:
results = await evaluate_grouped_summaries(
    grouped_news,
    summaries)

2026-04-02 22:52:04,164 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-02 22:52:06,363 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-02 22:52:08,903 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-02 22:52:13,088 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-02 22:52:16,097 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-02 22:52:18,333 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-02 22:52:20,092 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-02 22:52:22,244 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-02 22:52:24,698 | INFO |

In [39]:
results[1][0]

DayMetrics(date='2026-01-01', relevance_score=7.5, faithfulness_score=0.0, redundancy_score='Все новости уникальны и описывают различные аспекты текущей экономической ситуации и ожиданий на финансовых рынках. Хотя некоторые темы пересекаются, такие как ожидания экономических данных и стабильность рынков, они рассматриваются с разных точек зрения и не являются дублирующими.', relevance_reasoning='Выбранные 10 новостей охватывают ключевые аспекты мировых финансовых рынков и экономики, такие как закрытие рынков на Новый год, ожидания по экономическим данным, стабильность геополитической ситуации и корпоративные отчеты. Эти темы важны для понимания текущей ситуации на рынках и ожиданий на ближайшее будущее. Однако, учитывая, что это первый день нового года, возможно, не все из них являются самыми актуальными событиями дня, так как активность минимальна и значительных изменений не происходит. Тем не менее, они дают хорошее представление о том, чего ожидать в ближайшие недели.', faithfulness

In [40]:
results[1][0].faithfulness_score

0.0

In [41]:
results[1][0].relevance_score

7.5

In [42]:
results[1][0].redundancy_score

'Все новости уникальны и описывают различные аспекты текущей экономической ситуации и ожиданий на финансовых рынках. Хотя некоторые темы пересекаются, такие как ожидания экономических данных и стабильность рынков, они рассматриваются с разных точек зрения и не являются дублирующими.'

In [76]:
model_to_judge_results = {}

for model in models:
    results = await evaluate_grouped_summaries(
        grouped_news,
        model_to_summaries[model]
    )
    model_to_judge_results[model] = results[1]

2026-04-03 18:08:54,702 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-03 18:08:56,546 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-03 18:08:59,923 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-03 18:09:03,305 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-03 18:09:06,578 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-03 18:09:09,038 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-03 18:09:12,518 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-03 18:09:16,515 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-03 18:09:20,093 | INFO |

In [77]:
with open('judgeV2.pickle', 'wb') as f:
    pickle.dump(model_to_judge_results, f)

In [20]:
import numpy as np

In [56]:
with open('judge.pickle', 'rb') as f:
    model_to_judge_results1 = pickle.load(f)

In [ ]:


for model in models:
    model_to_judge_results1

In [62]:
model_to_judge_results_all

{'gpt-4.1-nano': [DayMetrics(date='2026-01-01', relevance_score=7.5, faithfulness_score=0.0, redundancy_score=10.0, relevance_reasoning='Выбранные 10 новостей охватывают ключевые аспекты, влияющие на мировые финансовые рынки, такие как закрытие рынков на Новый год, монетарная политика центральных банков, геополитическая стабильность, экономические показатели, корпоративные отчеты и валютные рынки. Эти темы действительно важны для понимания текущей ситуации на рынках. Однако, учитывая, что это начало года и многие рынки закрыты, некоторые из новостей могут не иметь немедленного влияния на рынки, что снижает их актуальность в данный момент. Также отсутствуют новости о конкретных событиях или изменениях, которые могли бы существенно повлиять на рынки в краткосрочной перспективе.', faithfulness_reasoning='ТОП-10 содержит множество фактов, которых не было в исходных данных. Исходные данные содержат только информацию о том, что в большинстве стран мира торгов нет из-за Нового года. В ТОП-10 

In [54]:
model_to_judge_results[model]

[DayMetrics(date='2026-02-01', relevance_score=8.5, faithfulness_score=10.0, redundancy_score=10.0, relevance_reasoning='Выбранные 10 новостей охватывают широкий спектр важных тем, включая геополитические события, экономические и финансовые рынки, а также энергетическую безопасность. Эти темы имеют значительное влияние на глобальную экономику и политику. Например, санкции против криптобирж и переговоры с Ираном могут существенно повлиять на рынок криптовалют и нефти. Также важны новости о запасах газа в Европе и решении ОПЕК+, так как они касаются энергетической безопасности и цен на нефть. Однако некоторые темы, такие как ситуация с долгом Oracle и призыв Си Цзиньпина сделать юань резервной валютой, могут быть менее актуальны в краткосрочной перспективе. В целом, выбор тем достаточно сбалансирован и отражает ключевые события дня.', faithfulness_reasoning='Все факты в ТОП-10 строго соответствуют исходным данным. Нет искажений, ошибок в именах, цифрах или выдуманных фактов. Каждое утвер

In [78]:
for model in models:
    print(model)
    relevance_scores = [d.relevance_score for d in model_to_judge_results[model]]
    faithfulness_scores = [d.faithfulness_score for d in model_to_judge_results[model]]

    print(np.mean(relevance_scores[1:]), relevance_scores)
    print(np.mean(faithfulness_scores[1:]), faithfulness_scores)

    print()


gpt-4.1-nano
8.464912280701755 [7.0, 8.5, 9.0, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.0, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.0, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.0, 8.5, 8.5, 8.0, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.0, 8.5, 8.5, 8.5]
7.631578947368421 [0.0, 8.0, 10.0, 8.0, 8.0, 5.0, 8.0, 8.0, 9.0, 8.0, 8.0, 8.0, 8.0, 9.0, 10.0, 8.0, 10.0, 8.0, 10.0, 0.0, 8.0, 8.0, 0.0, 8.0, 8.0, 0.0, 8.0, 8.0, 10.0, 8.0, 8.0, 8.0, 10.0, 8.0, 7.0, 0.0, 8.0, 8.0, 8.0, 0.0, 10.0, 9.0, 0.0, 8.0, 6.0, 10.0, 10.0, 10.0, 8.0, 10.0, 10.0, 8.0, 10.0, 10.0, 8.0, 8.0, 8.0, 10.0]

gpt-4.1-mini
8.491228070175438 [0.0, 8.5, 9.0, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.0, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.0, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5, 8.5]
9.228070175438596 [0.0, 10.0, 1

In [36]:
model_to_judge_results[models[1]]

[DayMetrics(date='2026-01-01', relevance_score=0.0, faithfulness_score=0.0, redundancy_score=0.0, relevance_reasoning='', faithfulness_reasoning='', redundancy_reasoning='', skipped=True, skip_reason='empty news_texts or summary'),
 DayMetrics(date='2026-01-02', relevance_score=8.5, faithfulness_score=10.0, redundancy_score=10.0, relevance_reasoning='Выбранные 10 новостей охватывают широкий спектр важных тем, включая экономические показатели, энергетический рынок, валютные колебания и достижения в автомобильной промышленности. Эти темы имеют значительное влияние на глобальные рынки и экономику. Однако, некоторые важные события, такие как изменения в крипторегулировании и геополитические события, не были включены в итоговый топ-10, что могло бы сделать обзор более полным. Тем не менее, выбранные темы действительно отражают ключевые экономические и рыночные тренды.', faithfulness_reasoning='Все факты в ТОП-10 строго соответствуют исходным данным. Нет искажений, ошибок в именах, цифрах ил

In [34]:
import datetime

In [35]:
grouped_news[datetime.date(2026, 1, 1)]

[NewsItem(message_id=353560, text='🎄**🗓КАЛЕНДАРЬ НА СЕГОДНЯ — 2026.01.01**\nВ большинстве стран мира торгов нет - Новый год', date=datetime.datetime(2026, 1, 1, 9, 11, 20, tzinfo=datetime.timezone.utc), views=78060, forwards=375)]

In [28]:
model_to_summaries[models[0]]

{datetime.date(2026, 1, 1): Summary(date=datetime.date(2026, 1, 1), news_count=1, key_topics=['Мировые финансовые рынки закрыты в связи с празднованием Нового года, активность минимальна.', 'Центральные банки большинства стран начали новый год без значительных изменений в монетарной политике, ожидается постепенное восстановление после предыдущих повышений ставок.', 'Геополитическая ситуация остается стабильной, что способствует умеренному оптимизму на рынках в начале 2026 года.', 'Экономические показатели за последние месяцы показывают признаки замедления роста в некоторых развитых странах, что вызывает осторожность у инвесторов.', 'Крупные корпорации готовятся к отчетным сезонам, ожидается публикация финансовых результатов за последний квартал 2025 года.', 'Важных политических событий, способных существенно повлиять на рынки, на сегодняшний день не зафиксировано.', 'Мировые валютные рынки демонстрируют низкую волатильность из-за отсутствия важных новостей и торговых операций.', 'Рынки

In [93]:
import numpy as np
from scipy import stats

# Предположим, мы сравниваем всех с этой моделью (baseline)
baseline_model = "gpt-4.1-nano"
baseline_relevance = [d.faithfulness_score for d in model_to_judge_results_all[baseline_model]]
baseline_relevance = baseline_relevance[1:]

print(f"Comparing all models against baseline: {baseline_model} {np.mean(baseline_relevance)}\n")

for model in models:
    # if model == baseline_model:
    #     continue
        
    results = model_to_judge_results[model]
    if not results: # Пропускаем пустые модели (как твои gpt-5)
        print(f"{model}: No data")
        continue

    results = [r for r in results if r.date != '2026-01-31']
    relevance_scores = [d.faithfulness_score for d in results]
    relevance_scores = relevance_scores[1:]
    
    # Проверяем, совпадает ли длина выборок для корректного теста
    if len(relevance_scores) == len(baseline_relevance):
        # Используем тест Уилкоксона для связных выборок
        stat, p_value = stats.wilcoxon(baseline_relevance, relevance_scores)
        
        mean_score = np.mean(relevance_scores)
        is_significant = "СУЩЕСТВЕННО" if p_value < 0.05 else "незначительно"
        
        print(f"--- {model} ---")
        print(f"Mean: {mean_score:.3f} (p-value: {p_value:.4f})")
        print(f"Разница с baseline {is_significant}")
    else:
        print(f"{model}: Ошибка (разная длина выборок)", len(relevance_scores), len(baseline_relevance))
    print()

Comparing all models against baseline: gpt-4.1-nano 7.071428571428571

--- gpt-4.1-nano ---
Mean: 7.625 (p-value: 0.1312)
Разница с baseline незначительно

--- gpt-4.1-mini ---
Mean: 9.250 (p-value: 0.0000)
Разница с baseline СУЩЕСТВЕННО

--- gpt-4o-mini ---
Mean: 8.884 (p-value: 0.0000)
Разница с baseline СУЩЕСТВЕННО



In [90]:
model_to_judge_results_all[model][0]

DayMetrics(date='2026-01-01', relevance_score=0.0, faithfulness_score=0.0, redundancy_score=0.0, relevance_reasoning='', faithfulness_reasoning='', redundancy_reasoning='', skipped=True, skip_reason='empty news_texts or summary')

In [67]:
## RElevance
Comparing all models against baseline: gpt-4.1-nano 8.169642857142858

--- gpt-4.1-mini ---
Mean: 8.482 (p-value: 0.1597)
Разница с baseline незначительно

--- gpt-4o-mini ---
Mean: 8.402 (p-value: 0.5232)
Разница с baseline незначительно

SyntaxError: invalid syntax (3290426693.py, line 2)

In [65]:
## faithfulness

Comparing all models against baseline: gpt-4.1-nano 7.071428571428571

--- gpt-4.1-mini ---
Mean: 9.071 (p-value: 0.0000)
Разница с baseline СУЩЕСТВЕННО

--- gpt-4o-mini ---
Mean: 9.125 (p-value: 0.0000)
Разница с baseline СУЩЕСТВЕННО

SyntaxError: invalid syntax (3536645786.py, line 3)

In [92]:
## RELEVANCE V2
Comparing all models against baseline: gpt-4.1-nano 8.169642857142858

--- gpt-4.1-nano ---
Mean: 8.464 (p-value: 0.4170)
Разница с baseline незначительно

--- gpt-4.1-mini ---
Mean: 8.491 (p-value: 0.1206)
Разница с baseline незначительно

--- gpt-4o-mini ---
Mean: 8.268 (p-value: 0.5157)
Разница с baseline незначительно

SyntaxError: invalid syntax (1704922813.py, line 2)

In [94]:
# Faithfulness V2
Comparing all models against baseline: gpt-4.1-nano 7.071428571428571

--- gpt-4.1-nano ---
Mean: 7.625 (p-value: 0.1312)
Разница с baseline незначительно

--- gpt-4.1-mini ---
Mean: 9.250 (p-value: 0.0000)
Разница с baseline СУЩЕСТВЕННО

--- gpt-4o-mini ---
Mean: 8.884 (p-value: 0.0000)
Разница с baseline СУЩЕСТВЕННО

SyntaxError: invalid syntax (507135208.py, line 2)